In [1]:
import sys, os
import numpy as np
from datetime import date
# import matplotlib.pyplot as plt

from xgboost import XGBClassifier
# from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score

sys.path.append('..')
from data import Data
from options import Options
from xgbooster import XGBooster

/Users/huhao_lleida/Workplace/code_Postdoc_Lleida/ExplainingBoostedRules/Exp3_BoostRules_MNIST/../xgbooster/xgbooster.py:543: SyntaxWarning: invalid escape sequence '\h'
  f.write("{} & {} & {} &{}  &{} & {} \\\\ \n \hline \n".format(


In [2]:
## 2025/7/16 => function to extract the identical rules used in all boosted trees
import re
from collections import deque

def extract_info_tree_text_line(line, node_type):
    # node_type = 0 => internal node; node_type=1 => leaf node
    # DEBUG: considering the positive & negative for internal node
    assert node_type == 0 or node_type == 1
    line_v = line.split(':')
    assert len(line_v) == 2
    node_index = int(line_v[0])
    if node_type == 1:
        line_value = line_v[-1].split('=')[-1]
        line_dir_index = None
    if node_type == 0:
        line_value = line_v[-1].split(' ')[0]
        # only consider yes / no
        line_dirs = line_v[-1].split(' ')[-1].split(',')[:-1] 
        assert len(line_dirs) == 2
        line_dir_index = [-1, -1]
        for l_d in line_dirs:
            if l_d.split('=')[0] == 'yes':
                line_dir_index[0] = int(l_d.split('=')[-1])
            else:
                line_dir_index[1] = int(l_d.split('=')[-1])
    return node_index, line_value, line_dir_index
      
def dfs_tree_path(tree_text_lines, tree_index, exist_paths):
    '''
        dfs approach to iterate all paths in a given tree
    '''
    stack = deque()
    t_paths_name, t_paths = [], []
    for t_line in tree_text_lines:
        tab_level = len(t_line) - len(t_line.strip('\t'))
        while(len(stack) > tab_level):
            stack.pop()
        
        if 'leaf' in t_line:
            curr_path = []
            curr_path_name = 'path_t' + str(tree_index) + '['
            # need extract the current path, without push the leaf node
            last_dir = None
            for n in stack:
                n_index, l_value, l_dir = extract_info_tree_text_line(n, 0)
                curr_path_name += str(n_index)
                if (not last_dir is None) and (last_dir.index(n_index) == 1):
                    # judge the negation by the node_index
                    curr_path[-1] = '>='.join(curr_path[-1].split('<'))
                last_dir = l_dir
                curr_path.append(l_value)
        
            
            n_index, l_value, _ = extract_info_tree_text_line(t_line.strip('\t'), 1)
            curr_path_name += str(n_index) + ']:'
            if (not last_dir is None) and (last_dir.index(n_index) == 1):
                curr_path[-1] = '>='.join(curr_path[-1].split('<'))
            curr_path = ''.join(sorted(curr_path))
            # judge curr_path is identical or not
            if not (curr_path in exist_paths):
                # extract the leaf node value
                t_paths.append(''.join(curr_path))
                t_paths_name.append(curr_path_name)
            continue
        stack.append(t_line.strip('\t'))
    
    while(len(stack) > 0):
        stack.pop()
    assert len(stack) == 0
    return t_paths_name, t_paths

def extract_identical_paths_from_boosted_trees(booster_fpath, out_fpath):
    '''
        Input: the file path of boosted trees stored (in the default format of XGBoost package)
        Output: the txt containing all identical paths => same format as the boosted rules stored by BOOMER
    '''
    assert os.path.isfile(booster_fpath)
    identical_path_names, identical_paths = [], []

    with open(booster_fpath, 'r') as f:
        tree_index = -1
        tree_text = ''
        for line in f.readlines():
            if line.startswith('booster'):
                if len(tree_text) > 0:
                    t_text_lines = tree_text.split('\n')
                    t_paths_name, t_paths = dfs_tree_path(t_text_lines, tree_index, identical_paths)
                    identical_path_names += t_paths_name
                    identical_paths += t_paths
                    tree_text = ''
                # extract the index of tree 
                tree_index = int(re.search(r'\[(.*?)\]', line).group(1))
            else:
                tree_text += line
    
    # need to store the extracted paths into the out_fpath
    assert (len(identical_path_names) == len(identical_paths))
    with open(out_fpath, 'w') as f:
        for i in range(len(identical_paths)):
            f.write(identical_path_names[i] + '\n')
            f.write('{' + identical_paths[i] + '}\n')
    return len(identical_paths)

def train_specific_xgb(dataset, n_estimator, max_depth, output, wdigit=None):
    # preparing the command for options
    if not wdigit is None:
        command_line = 'xxx -t -n ' + str(n_estimator) + ' -d ' + str(max_depth) + ' --wdigit ' + str(wdigit) + ' -o ' + output + ' ' + dataset
    else:
        command_line = 'xxx -t -n ' + str(n_estimator) + ' -d ' + str(max_depth) + ' -o ' + output + ' ' + dataset
    options = Options(command_line.split())

    if options.files:
        xgb = None
        if options.train:
            data = Data(filename=options.files[0], mapfile=options.mapfile,
                    separator=options.separator,
                    use_categorical=options.use_categorical)
            xgb = XGBooster(options, from_data=data)
            train_acc, test_acc, mod_fpath = xgb.train()
            return train_acc, test_acc, mod_fpath, xgb.nb_features, xgb.num_class, xgb.num_instance


In [ ]:
# 251011: prepare the name of the output folder
xgboost_out = 'mnist_xgboost_' + date.today().isoformat()[2:].replace('-', '')

# The default split ration is 20% for testing accuracy
nb_candidate = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 150]
result_json = []
max_conds = [4, 5, 6]
dataset = '../bench_mnist_csv/mnist_train.csv'

for nb in nb_candidate:
    for max_cond in max_conds:
        
        train_acc, test_acc, mod_fpath, nb_feat, nb_c, nb_i = train_specific_xgb(dataset, nb, max_cond, xgboost_out)
        
        mod_tpath_fpath = mod_fpath[:-3] + 'path'
        if os.path.exists(mod_tpath_fpath):
            os.remove(mod_tpath_fpath)
        num_identical_paths = extract_identical_paths_from_boosted_trees(mod_fpath, mod_tpath_fpath)
        print("dataset {}, nb_trees {}, max_con {}, n_unique {}".format(dataset, nb, max_cond, num_identical_paths))

        dataset_n = dataset.split('/')[-1].split('.')[0]
        res_dict = {'dataset': dataset_n, 'n_estimator': nb, 'max_depth': max_cond,\
                    'train_acc': train_acc, 'test_acc': test_acc, 'num_feat': nb_feat,\
                    'num_class': nb_c, 'num_instance': nb_i, 'num_identical_path': num_identical_paths}
        result_json.append(res_dict)

In [3]:
# 260304: the wdigit condition
wdigits = [1, 2]

xgboost_out = 'mnist_xgboost_' + date.today().isoformat()[2:].replace('-', '')
# The default split ration is 20% for testing accuracy
nb_candidate = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
result_json = []
max_conds = [4, 5, 6]
dataset = '../bench_mnist_csv/mnist_train.csv'

for nb in nb_candidate:
    for max_cond in max_conds:
        for wd in wdigits:
            train_acc, test_acc, mod_fpath, nb_feat, nb_c, nb_i = train_specific_xgb(dataset, nb, max_cond, xgboost_out, wd)
            
            mod_tpath_fpath = mod_fpath[:-3] + 'path'
            if os.path.exists(mod_tpath_fpath):
                os.remove(mod_tpath_fpath)
            num_identical_paths = extract_identical_paths_from_boosted_trees(mod_fpath, mod_tpath_fpath)
            print("dataset {}, nb_trees {}, max_con {}, n_unique {}".format(dataset, nb, max_cond, num_identical_paths))

            dataset_n = dataset.split('/')[-1].split('.')[0]
            res_dict = {'dataset': dataset_n, 'n_estimator': nb, 'max_depth': max_cond,\
                        'train_acc': train_acc, 'test_acc': test_acc, 'num_feat': nb_feat,\
                        'num_class': nb_c, 'num_instance': nb_i, 'num_identical_path': num_identical_paths,\
                        'wdigit': wd}
            result_json.append(res_dict)


saving results to  mnist_xgboost_260304/mnist_train/mnist_train_nbestim_10_maxdepth_4_testsplit_0.2_wd_1.res.txt
dataset ../bench_mnist_csv/mnist_train.csv, nb_trees 10, max_con 4, n_unique 1561
saving results to  mnist_xgboost_260304/mnist_train/mnist_train_nbestim_10_maxdepth_4_testsplit_0.2_wd_2.res.txt
dataset ../bench_mnist_csv/mnist_train.csv, nb_trees 10, max_con 4, n_unique 1561
saving results to  mnist_xgboost_260304/mnist_train/mnist_train_nbestim_10_maxdepth_5_testsplit_0.2_wd_1.res.txt
dataset ../bench_mnist_csv/mnist_train.csv, nb_trees 10, max_con 5, n_unique 3015
saving results to  mnist_xgboost_260304/mnist_train/mnist_train_nbestim_10_maxdepth_5_testsplit_0.2_wd_2.res.txt
dataset ../bench_mnist_csv/mnist_train.csv, nb_trees 10, max_con 5, n_unique 3015
saving results to  mnist_xgboost_260304/mnist_train/mnist_train_nbestim_10_maxdepth_6_testsplit_0.2_wd_1.res.txt
dataset ../bench_mnist_csv/mnist_train.csv, nb_trees 10, max_con 6, n_unique 5482
saving results to  mnist_

In [4]:
import json
json_file_name = 'result' + xgboost_out[4:] + '.json'
if os.path.isfile(json_file_name):
    os.remove(json_file_name)
with open(json_file_name, 'w') as f:
    json.dump(result_json, f, indent=2)